In [1]:
from bs4 import BeautifulSoup as bs
import requests
import pandas as pd

In [2]:
url = 'https://www.kinonews.ru/top250imdb/'
st_accept = "text/html"
st_useragent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:128.0) Gecko/20100101 Firefox/128.0"
# формируем хеш заголовков
session = requests.Session()
session.headers.update({
   "Accept": st_accept,
   "User-Agent": st_useragent,
    'x-test': 'true'
})
session.auth = ('username', 'password')
page = session.get(url)
page.encoding = 'utf-8'

In [3]:
page.status_code

200

In [4]:
soup = bs(page.text, 'lxml')

In [5]:
soup

<!DOCTYPE html>
<html lang="ru">
<head>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="width=device-width, initial-scale=1, user-scalable=yes" name="viewport"/>
<link href="https://www.kinonews.ru/favicon.ico" rel="icon" type="image/x-icon"/>
<link href="https://www.kinonews.ru/favicon.ico" rel="shortcut icon" type="image/x-icon"/>
<meta content="KinoNews.ru" property="twitter:domain"/>
<meta content="summary_large_image" property="twitter:card"/>
<meta content="@KinoNewsRu" property="twitter:site"/>
<meta content="@KinoNewsRu" name="twitter:creator"/>
<meta content="summary_large_image" name="twitter:card"/>
<title>Топ 250 лучших фильмов по версии портала о кино IMDb</title>
<meta content="Топ 250 лучших фильмов по версии портала о кино IMDb" name="title"/>
<meta content="топ 250 фильмов, топ 250 imdb, топ 250 лучших фильмов, топ 250 фильмов imdb, рейтинг imdb топ 250" name="keywords"/>
<meta content="Список 250 лучших фильмов за всю историю мирово

In [6]:
result_list = {'title': [], 'release_date': [], 'duration': [], 'country': [], 'genre': []}

In [7]:
pagenum = 1

for i in range(1,6):
    if i == 1:
        url = 'https://www.kinonews.ru/top250imdb/'
    else:
        url = f'https://www.kinonews.ru/top250imdb_p{pagenum}/' 
    page = session.get(url)
    page.encoding = 'utf-8'
    soup = bs(page.text, 'html.parser')
    titles = soup.find_all('a', class_='titlefilm')
    release_dates = soup.find_all('div', class_='bigtext')
    durations = soup.find_all('div', class_='rating_rightdesc')
    countries = soup.find_all('div', class_='textgray')
    genres = soup.find_all('div', class_='textgray')
    pagenum += 1

    for j in titles:
        result_list['title'].append(j.text)
        print(j.text)
        film_url = 'https://www.kinonews.ru' + j.get('href')
        _page = requests.get(film_url)
        film_soup = bs(_page.text, 'html.parser')
        desc = film_soup.find('div', class_='game_main').find('div', class_='textart15')
        print(desc.text.replace('\n',''))
        result_list['description'].append(desc.text.replace('\n',''))
        print('\n\n')
    for j in release_dates:
        if j.text.split(' ')[-1].isdigit():
            result_list['release_date'].append(int(j.text.split(' ')[-1]))
    for j in durations:
        if len(j.text.split('\n')) == 7:
            result_list['duration'].append(j.text.split('\n')[5].split(': ')[1])
        elif len(j.text.split('\n')) == 6:
            result_list['duration'].append(None)
    for j in countries:
        if j.text.split(': ')[0].__contains__('Страна'):
            result_list['country'].append(j.text.split(': ')[1])
    for j in countries:
        if j.text.split(': ')[0].__contains__('Жанр'):
            result_list['genre'].append(j.text.split(': ')[1])

Побег из Шоушенка
"Побег из Шоушенка" - фильм, который считается одним из лучших в истории кино. Режиссером выступил Фрэнк Дарабонт, а в главных ролях снялись Тим Роббинс и Морган Фриман. Фильм основан на рассказе Стивена Кинга и повествует о бухгалтере Энди Дюфрейне, который осужден за убийство жены и ее любовника. Он отправляется в тюрьму Шоушенк, где сталкивается со многими трудностями и проблемами, но не теряет надежды на свободу. "Побег из Шоушенка" провалился в мировом прокате, был номинирован на несколько премий "Оскар", включая лучший фильм, лучшую режиссуру и лучший сценарий, но не получил ни одной премии. Однако с течением времени фильм "Побег из Шоушенка" стал считаться классикой кинематографа и завоевал множество наград и признаний. В целом, "Побег из Шоушенка" - потрясающий фильм, который оставит незабываемые впечатления у зрителя. Фильм заслуженно стал классикой кинематографа и продолжает радовать зрителей своей глубиной.


KeyError: 'description'

In [ ]:
file_name = 'parse_data.csv'
df = pd.DataFrame(data=result_list)
df.to_csv(file_name)

In [ ]:
df.head()

In [ ]:
df.info()